# Классификация ботов: Precision@Recall>=0.7

Bot detection (Avito): финальное решение.
Python 3.12.7, pandas 2.3.3, numpy 2.3.3, scikit-learn 1.8.0. Только классический ML, локально.
Запуск из корня архива (рядом data/, metric.py, sample_submission.csv):  python3 solution_final.py

Пайплайн:
  1. Очистка событий: дубли, сортировка, строгое окно [window_start, window_end).
  2. Признаки по группам: volume, timing, timing2, diversity, mix, pointer, pointer2, seq,
     sess, cross, ua, age. cross считается по всем кукам (train+test) БЕЗ target.
  3. Валидация по времени: два сплита (окна >= SPLIT_A и >= SPLIT_B), официальная metric.py.
  4. Кандидаты (набор групп x бленд) сравниваются; выбирается лучший по среднему PR-AUC на
     двух сплитах (PR-AUC устойчивее, чем P@R на ~160 ботах).
  5. Финал: обучение на всём train, 5 сидов, submission.csv + results.csv для README.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score

from metric import precision_at_recall

SEED = 42
TS = ["cookie_created_at", "window_start_ts", "window_end_ts"]
SPLIT_A, SPLIT_B = "2026-04-17", "2026-04-14"    # поправь B, если в train нет таких окон

train = pd.read_csv("data/train.csv", parse_dates=TS)
test = pd.read_csv("data/test.csv", parse_dates=TS)
events = pd.read_csv("data/events.csv.gz", parse_dates=["event_ts"])
sample = pd.read_csv("sample_submission.csv")
y = train.target.values

print("train windows:", train.window_start_ts.min(), "->", train.window_start_ts.max())
val = {"A": train.window_start_ts.ge(SPLIT_A).values, "B": train.window_start_ts.ge(SPLIT_B).values}
for k, m in val.items():
    print(f"split {k}: valid={m.sum()} bots={y[m].sum()} share={y[m].mean():.4f}")

## 1. Очистка событий
Дубли, привязка к окну куки, строгое окно [start, end), стабильная сортировка.

In [ ]:
def clean_events(events, meta):
    """Дедупликация, привязка к окну куки, строгое окно, стабильная сортировка."""
    e = events.drop_duplicates()
    e = e.merge(meta[["cookie_id", "window_start_ts", "window_end_ts"]], on="cookie_id")
    e = e[(e.event_ts >= e.window_start_ts) & (e.event_ts < e.window_end_ts)]
    return e.sort_values(["cookie_id", "event_ts"], kind="stable").reset_index(drop=True)

## 2. Признаки по кукам
Группы: volume, timing, timing2, sess, diversity, mix, pointer, pointer2, seq, ua, age.

In [ ]:
def build_features(meta, events, pairs=None):
    e = clean_events(events, meta)
    g = e.groupby("cookie_id")
    m = meta.set_index("cookie_id")
    F = {}

    F["volume"] = pd.DataFrame({"n_events": g.size(), "item_nunique": g.item_id.nunique()})
    n_ev = F["volume"].n_events

    # timing: интервалы между событиями
    e["gap"] = g.event_ts.diff().dt.total_seconds()
    gg = e.groupby("cookie_id").gap
    t = pd.DataFrame({"gap_mean": gg.mean(), "gap_std": gg.std(), "gap_min": gg.min(),
                      "gap_med": gg.median(), "gap_q10": gg.quantile(0.1), "gap_q90": gg.quantile(0.9)})
    t["gap_cv"] = t.gap_std / t.gap_mean.clip(lower=1e-6)
    e["gap_r"] = e.gap.round(1)
    top = e.groupby(["cookie_id", "gap_r"]).size().groupby(level=0).max()
    t["gap_mode_share"] = top / gg.count().clip(lower=1)             # доля самого частого интервала
    t["gap_lt1s_share"] = e.assign(x=e.gap < 1).groupby("cookie_id").x.mean()
    minute = e.event_ts.dt.floor("min")
    t["max_events_per_min"] = e.groupby(["cookie_id", minute]).size().groupby(level=0).max()
    t["active_minutes"] = e.assign(m=minute).groupby("cookie_id").m.nunique()
    t["active_hours"] = e.assign(h=e.event_ts.dt.floor("h")).groupby("cookie_id").h.nunique()
    t["night_share"] = e.assign(x=e.event_ts.dt.hour.between(0, 5)).groupby("cookie_id").x.mean()
    t["span_s"] = g.event_ts.agg(lambda s: (s.max() - s.min()).total_seconds())
    F["timing"] = t

    # timing2: регулярность, автокорреляция, «машинные» таймстемпы, оффсеты
    t2 = pd.DataFrame({"gap_q25": gg.quantile(0.25), "gap_q75": gg.quantile(0.75),
                       "gap_max": gg.max(), "gap_skew": gg.skew()})
    t2["burstiness"] = (t.gap_std - t.gap_mean) / (t.gap_std + t.gap_mean).clip(lower=1e-6)
    t2["gap_iqr_rel"] = (t2.gap_q75 - t2.gap_q25) / t.gap_med.clip(lower=1e-6)
    e["log_gap"] = np.log1p(e.gap.clip(lower=0))
    t2["log_gap_std"] = e.groupby("cookie_id").log_gap.std()
    e["gap_prev"] = e.groupby("cookie_id").gap.shift()
    a = e.dropna(subset=["gap", "gap_prev"]).assign(
        xz=lambda d: d.gap * d.gap_prev, x2=lambda d: d.gap ** 2, z2=lambda d: d.gap_prev ** 2)
    am = a.groupby("cookie_id")[["gap", "gap_prev", "xz", "x2", "z2"]].mean()
    den = np.sqrt(((am.x2 - am.gap ** 2) * (am.z2 - am.gap_prev ** 2)).clip(lower=1e-12))
    t2["gap_autocorr1"] = (am.xz - am.gap * am.gap_prev) / den
    t2["zero_us_share"] = e.assign(x=e.event_ts.dt.microsecond == 0).groupby("cookie_id").x.mean()
    e["whole_gap"] = (e.gap.round(6) % 1 == 0).where(e.gap.notna())
    t2["whole_gap_share"] = e.groupby("cookie_id").whole_gap.mean()
    idx = g.size().index
    t2["first_offset_s"] = (g.event_ts.min() - m.window_start_ts.reindex(idx)).dt.total_seconds()
    t2["last_offset_s"] = (m.window_end_ts.reindex(idx) - g.event_ts.max()).dt.total_seconds()
    t2 = t2.join(e.groupby(["cookie_id", "event_name"]).gap.median().unstack().add_prefix("gapmed_"))
    F["timing2"] = t2

    # sess: сессии по паузе > 30 минут
    e["new_sess"] = (e.gap.isna() | (e.gap > 1800)).astype(int)
    e["sess_id"] = e.groupby("cookie_id").new_sess.cumsum()
    sg = e.groupby(["cookie_id", "sess_id"]).agg(n=("event_ts", "size"), t0=("event_ts", "min"),
                                                 t1=("event_ts", "max"))
    sg["dur"] = (sg.t1 - sg.t0).dt.total_seconds()
    F["sess"] = sg.groupby("cookie_id").agg(n_sessions=("n", "size"), sess_len_mean=("n", "mean"),
                                            sess_len_max=("n", "max"), sess_dur_mean=("dur", "mean"))

    # diversity
    d = pd.DataFrame({"cat_nunique": g.item_category.nunique(), "loc_nunique": g.item_location.nunique(),
                      "query_nunique": g.search_query.nunique(),
                      "page_max": g.search_page.max(), "page_mean": g.search_page.mean()})
    d["item_ratio"] = F["volume"].item_nunique / n_ev
    d["query_ratio"] = d.query_nunique / n_ev
    d["item_repeat_share"] = 1 - F["volume"].item_nunique / g.item_id.count().clip(lower=1)
    F["diversity"] = d

    # mix: доли типов событий
    cnt = pd.crosstab(e.cookie_id, e.event_name)
    F["mix"] = cnt.div(cnt.sum(axis=1), axis=0).add_prefix("share_")

    # pointer и pointer2 (по платформам, шаги курсора)
    F["pointer"] = pd.DataFrame({
        "ptr_missing": g.pointer_x.apply(lambda s: s.isna().mean()),
        "ptr_x_std": g.pointer_x.std(), "ptr_y_std": g.pointer_y.std(),
        "ptr_pairs_nunique": e.dropna(subset=["pointer_x"]).assign(
            p=lambda x: x.pointer_x.astype(str) + "_" + x.pointer_y.astype(str)
        ).groupby("cookie_id").p.nunique()})
    miss = e.assign(mi=e.pointer_x.isna()).groupby(["cookie_id", "platform"]).mi.mean().unstack()
    p2 = miss.add_prefix("ptr_miss_").join(
        pd.crosstab(e.cookie_id, e.platform, normalize="index").add_prefix("plat_"))
    dx = e.pointer_x - e.groupby("cookie_id").pointer_x.shift()
    dy = e.pointer_y - e.groupby("cookie_id").pointer_y.shift()
    e["step"] = np.hypot(dx, dy)
    gs = e.groupby("cookie_id").step
    p2["step_mean"], p2["step_med"] = gs.mean(), gs.median()
    p2["step_zero_share"] = e.assign(z=e.step == 0).where(e.step.notna()).groupby("cookie_id").z.mean()
    p2["ptr_x0_share"] = e.assign(z=e.pointer_x == 0).where(e.pointer_x.notna()).groupby("cookie_id").z.mean()
    F["pointer2"] = p2

    # seq: переходы между событиями, листание выдачи
    e["prev_ev"] = e.groupby("cookie_id").event_name.shift()
    e["pair"] = e.prev_ev + ">" + e.event_name
    if pairs is None:                                   # топ-пары берём по train, для test переиспользуем
        pairs = list(e.pair.value_counts().head(15).index)
    pc = pd.crosstab(e.cookie_id, e.pair).reindex(columns=pairs, fill_value=0)
    s = pc.div(n_ev.reindex(pc.index), axis=0).add_prefix("pair_")
    s.columns = [c.replace(" ", "_") for c in s.columns]
    dp = e.search_page - e.groupby("cookie_id").search_page.shift()
    s["page_step1_share"] = e.assign(x=dp == 1).where(dp.notna()).groupby("cookie_id").x.mean()
    s["page_same_share"] = e.assign(x=dp == 0).where(dp.notna()).groupby("cookie_id").x.mean()
    s["page_back_share"] = e.assign(x=dp < 0).where(dp.notna()).groupby("cookie_id").x.mean()
    F["seq"] = s

    # ua: структурные признаки, не сырая строка
    ua = e.user_agent.fillna("")
    e["ua_lib"] = ua.str.contains(r"python|requests|curl|okhttp|httpx|scrapy|go-http|aiohttp|axios", case=False)
    e["ua_headless"] = ua.str.contains("headless", case=False)
    e["ua_no_mozilla"] = ~ua.str.startswith("Mozilla")
    e["chrome_v"] = pd.to_numeric(ua.str.extract(r"Chrome/(\d+)")[0], errors="coerce")
    gu = e.groupby("cookie_id")
    F["ua"] = pd.DataFrame({"ua_nunique": gu.user_agent.nunique(), "platform_nunique": gu.platform.nunique(),
                            "ua_lib": gu.ua_lib.mean(), "ua_headless": gu.ua_headless.mean(),
                            "ua_no_mozilla": gu.ua_no_mozilla.mean(),
                            "chrome_v_max": gu.chrome_v.max(), "chrome_v_min": gu.chrome_v.min()})

    F["age"] = pd.DataFrame({"cookie_age_h": (m.window_start_ts - m.cookie_created_at).dt.total_seconds() / 3600})

    out = {k: df.reindex(meta.cookie_id) for k, df in F.items()}
    out["volume"] = out["volume"].fillna({"n_events": 0, "item_nunique": 0})
    return out, pairs

## 3. Признаки "флотов" (cross)
Счётчики по всем кукам суточных окон (train+test), без target.

In [ ]:
def cross_features(meta_all, events):
    """Насколько «общие» у куки UA / объявления / запросы / секунды с другими куками того же дня."""
    e = clean_events(events, meta_all)
    e["day"] = e.window_start_ts.dt.date
    e["sec"] = e.event_ts.dt.floor("s")
    nc = lambda cols: e.groupby(["day"] + cols).cookie_id.transform("nunique")
    e["ua_n"] = nc(["user_agent"]).where(e.user_agent.notna())
    e["item_n"] = nc(["item_id"]).where(e.item_id.notna())
    e["query_n"] = nc(["search_query"]).where(e.search_query.notna())
    e["sync_n"] = nc(["sec"])
    g = e.groupby("cookie_id")
    return pd.DataFrame({
        "ua_share_mean": np.log1p(g.ua_n.mean()), "ua_share_max": np.log1p(g.ua_n.max()),
        "item_share_mean": np.log1p(g.item_n.mean()),
        "item_shared_frac": e.assign(x=e.item_n >= 5).where(e.item_n.notna()).groupby("cookie_id").x.mean(),
        "query_share_mean": np.log1p(g.query_n.mean()),
        "query_shared_frac": e.assign(x=e.query_n >= 3).where(e.query_n.notna()).groupby("cookie_id").x.mean(),
        "sync_frac": e.assign(x=e.sync_n > 1).groupby("cookie_id").x.mean(),
        "sync_mean": np.log1p(g.sync_n.mean())})


Ftr, PAIRS = build_features(train, events)
Fte, _ = build_features(test, events, pairs=PAIRS)
cross = cross_features(pd.concat([train, test], ignore_index=True), events)
Ftr["cross"], Fte["cross"] = cross.reindex(train.cookie_id), cross.reindex(test.cookie_id)
for k in Fte:                                            # одинаковые колонки и порядок в train/test
    Fte[k] = Fte[k].reindex(columns=Ftr[k].columns)
GROUPS = list(Ftr)
print({k: v.shape[1] for k, v in Ftr.items()})

assemble = lambda F, groups: pd.concat([F[k] for k in groups], axis=1)

## 4. Модели
HGB (регуляризация) и ExtraTrees, бленд по рангам.

In [ ]:
def hgb(seed):
    # сильная регуляризация: ~740 ботов в train, глубокие деревья переобучаются
    return HistGradientBoostingClassifier(max_depth=4, learning_rate=0.03, max_iter=600,
                                          min_samples_leaf=40, l2_regularization=3.0, random_state=seed)

def et(seed):
    return ExtraTreesClassifier(n_estimators=500, min_samples_leaf=2, max_features=0.5,
                                n_jobs=-1, random_state=seed)

def predict(X_tr, y_tr, X_va, blend, seeds):
    p = np.mean([hgb(s).fit(X_tr, y_tr).predict_proba(X_va)[:, 1] for s in seeds], axis=0)
    if not blend:
        return p
    q = et(SEED).fit(X_tr.fillna(-1), y_tr).predict_proba(X_va.fillna(-1))[:, 1]
    rk = lambda v: pd.Series(v).rank(pct=True).values
    return 0.5 * rk(p) + 0.5 * rk(q)

def evaluate(groups, blend):
    X, r = assemble(Ftr, groups), {}
    for k, mask in val.items():
        p = predict(X[~mask], y[~mask], X[mask], blend, seeds=(0, 1, 2))
        r[f"P@R_{k}"], r[f"PRAUC_{k}"] = precision_at_recall(y[mask], p), average_precision_score(y[mask], p)
    return r

## 5. Валидация по времени и выбор варианта
Два временных сплита, официальная metric.py, выбор по среднему PR-AUC.

In [ ]:
V2 = ["volume", "timing", "diversity", "mix", "pointer", "ua", "age"]
LEAN = [g for g in GROUPS if g not in ("volume", "ua", "age")]
cands = {
    "baseline (volume)": (["volume"], False),
    "v2 groups": (V2, False),
    "all": (GROUPS, False),
    "all + blend": (GROUPS, True),
    "lean": (LEAN, False),
    "lean + blend": (LEAN, True),
}
for g in ["cross", "sess", "timing2", "pointer2", "seq"]:
    cands[f"all minus {g}"] = ([x for x in GROUPS if x != g], False)

res = pd.DataFrame([{"setup": n, **evaluate(gr, bl)} for n, (gr, bl) in cands.items()])
res["mean_PRAUC"] = res[["PRAUC_A", "PRAUC_B"]].mean(axis=1)
print(res.round(4).to_string(index=False))
res.round(4).to_csv("results.csv", index=False)         # таблица для README

best = res.loc[res.mean_PRAUC.idxmax(), "setup"]         # выбор по среднему PR-AUC на двух сплитах
FINAL_GROUPS, BLEND = cands[best]
print("Выбран:", best)

## 6. Финальное обучение и submission.csv

In [ ]:
Xtr_f, Xte_f = assemble(Ftr, FINAL_GROUPS), assemble(Fte, FINAL_GROUPS)
score = predict(Xtr_f, y, Xte_f, blend=BLEND, seeds=range(5))

sub = pd.DataFrame({"cookie_id": test.cookie_id.values, "score": score})
assert list(sub.columns) == ["cookie_id", "score"]
assert len(sub) == len(test) and sub.cookie_id.is_unique and set(sub.cookie_id) == set(sample.cookie_id)
assert sub.score.notna().all() and sub.score.between(0, 1).all()
sub.to_csv("submission.csv", index=False)
print("OK", sub.shape)